Data Preparetion

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import io
import requests

data_url = "https://raw.github.com/BamlakHun/Memerekiya/main/Dataset_NLP.csv"

response = requests.get(data_url)
response.encoding = "utf-8"

df = pd.read_csv(
    io.StringIO(response.text),
    sep=";",
    engine="python",
    quotechar='"',
    doublequote=True,
    on_bad_lines="warn"
)

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(df.head(3).to_string())

Load Data

In [ ]:
pd.set_option('display.max_colwidth', None)
print(df.head(10).to_string())

In [ ]:
pd.set_option('display.max_colwidth', None)
print(df["Interview "].head(3).to_string())

# Implementation

In [ ]:
!pip install transformers torch sentence-transformers umap-learn hdbscan scikit-learn matplotlib seaborn -q

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

print("Loading KB-BERT.")

MODEL_NAME = "KB/bert-base-swedish-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Load all 37 interviews across 28 places

In [ ]:
import pandas as pd
import requests
import io

data_url = "https://raw.github.com/BamlakHun/Memerekiya/main/Dataset_NLP.csv"

response = requests.get(data_url)
response.encoding = "utf-8"

df = pd.read_csv(
    io.StringIO(response.text),
    sep=";",
    engine="python",
    quotechar='"',
    doublequote=True,
    on_bad_lines="warn"
)

df = df.rename(columns={
    df.columns[0]: "date",
    df.columns[1]: "place",
    df.columns[2]: "text",
    df.columns[3]: "gender",
    df.columns[4]: "age"
})

df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 50].reset_index(drop=True)

print(f"Loaded {len(df)} interviews across {df['place'].nunique()} places")
print(df["place"].value_counts())

*   Encode a single text with KB-BERT.
*   Uses mean pooling over all token embeddings — standard for sentence-level tasks.
*   Long texts are truncated to 512 tokens (KB-BERT's maximum).







In [ ]:
import numpy as np

def get_kbbert_embedding(text, tokenizer, model, max_length=512):

    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=True
    )

    # Forward pass, no gradient needed
    with torch.no_grad():
        outputs = model(**inputs)

    # Mean pooling: average all token embeddings
    # Shape: [1, seq_len, 768] → [768]
    token_embeddings = outputs.last_hidden_state.squeeze(0)
    attention_mask = inputs["attention_mask"].squeeze(0)
    mask_expanded = attention_mask.unsqueeze(-1).float()
    embedding = (token_embeddings * mask_expanded).sum(0) / mask_expanded.sum(0)

    return embedding.numpy()

# Generate embeddings for all interviews
print("Generating KB-BERT embeddings...")
embeddings = []

for i, row in df.iterrows():
    emb = get_kbbert_embedding(row["text"], tokenizer, model)
    embeddings.append(emb)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(df)} done")

embeddings = np.array(embeddings)
print(f"\nEmbedding matrix shape: {embeddings.shape}")
# Should be (n_interviews, 768) and it is

Average all interview embeddings per village
This gives one vector per village representing its collective character

In [ ]:
village_embeddings = {}
for place in df["place"].unique():
    mask = df["place"] == place
    village_embeddings[place] = embeddings[mask].mean(axis=0)

places = list(village_embeddings.keys())
village_matrix = np.array([village_embeddings[p] for p in places])

print(f"Village-level embedding matrix: {village_matrix.shape}")
print(f"Villages: {places}")

Use UMAP: reduce 768 dimensions → 2D for visualisation, n_neighbors controls how local vs global the structure is and for the interviews, keep n_neighbors small (5-10)

In [ ]:
import umap
import hdbscan


reducer = umap.UMAP(
    n_neighbors=5,
    min_dist=0.3,
    n_components=2,
    metric="cosine",   # cosine works better than euclidean for embeddings
    random_state=42
)

umap_2d = reducer.fit_transform(village_matrix)
print(f"UMAP output shape: {umap_2d.shape}")

# HDBSCAN clustering
# min_cluster_size=2 because you have few villages
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=2,
    min_samples=1,
    metric="euclidean"
)

cluster_labels = clusterer.fit_predict(umap_2d)
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)

print(f"\nClusters found: {n_clusters}")
print(f"Unclustered (noise) points: {(cluster_labels == -1).sum()}")
for place, label in zip(places, cluster_labels):
    tag = f"Cluster {label}" if label != -1 else "Unclustered"
    print(f"  {place}: {tag}")

# Visualization

Village clusters — KB-BERT embeddings (UMAP + HDBSCAN

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 8))

colors = ["#534AB7", "#0F6E56", "#BA7517", "#E24B4A",
          "#185FA5", "#7C3AED", "#059669", "#DC2626"]

unique_clusters = sorted(set(cluster_labels))

for cluster_id in unique_clusters:
    mask = cluster_labels == cluster_id
    color = "#AAAAAA" if cluster_id == -1 else colors[cluster_id % len(colors)]
    label = "Unclustered" if cluster_id == -1 else f"Cluster {cluster_id}"

    ax.scatter(
        umap_2d[mask, 0],
        umap_2d[mask, 1],
        c=color,
        s=180,
        alpha=0.85,
        edgecolors="white",
        linewidths=1.2,
        label=label,
        zorder=3
    )

# Label each village
for i, place in enumerate(places):
    ax.annotate(
        place,
        (umap_2d[i, 0], umap_2d[i, 1]),
        fontsize=9,
        xytext=(6, 6),
        textcoords="offset points",
        color="#333333"
    )

ax.set_title("Village clusters — KB-BERT embeddings (UMAP + HDBSCAN)",
             fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("UMAP dimension 1", fontsize=10)
ax.set_ylabel("UMAP dimension 2", fontsize=10)
ax.legend(fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.15)
ax.set_facecolor("#FAFAFA")

plt.tight_layout()
plt.savefig("kbbert_village_clusters.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: kbbert_village_clusters.png")

From the interview we can see which are most similar village, in pairs

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

sim_matrix = cosine_similarity(village_matrix)

# Convert to ranked pairs instead of a heatmap
pairs = []
for i in range(len(places)):
    for j in range(i+1, len(places)):
        pairs.append({
            "village_1": places[i],
            "village_2": places[j],
            "similarity": round(sim_matrix[i][j], 3)
        })

pairs_df = pd.DataFrame(pairs).sort_values("similarity", ascending=False)
print("Most similar village pairs:")
print(pairs_df.head(10).to_string(index=False))
print("\nMost distinct village pairs:")
print(pairs_df.tail(5).to_string(index=False))

Hierarchical clustering that shows which villages group together. Moreover at what distance, which is more informative than raw similarity scores

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cosine
import matplotlib.pyplot as plt


distance_matrix = 1 - cosine_similarity(village_matrix)
linkage_matrix = linkage(distance_matrix, method="ward")

fig, ax = plt.subplots(figsize=(24, 10))
dendrogram(
    linkage_matrix,
    labels=places,
    ax=ax,
    leaf_rotation=45,
    leaf_font_size=10,
    color_threshold=0.3
)

ax.set_title("Village groupings — KB-BERT hierarchical clustering",
             fontsize=12, fontweight="bold")
ax.set_ylabel("Distance (dissimilarity)", fontsize=10)
ax.set_xlabel("Village", fontsize=10)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.savefig("kbbert_dendrogram.png", dpi=150, bbox_inches="tight")
plt.show()

Instead of embedding whole interviews, embed sentence by sentence. This gives much richer topic signal on small data

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

def split_into_sentences(text):
    """Split Swedish interview text into sentences."""
    import re
    # Split on periods, question marks, newlines
    sentences = re.split(r'(?<=[.!?])\s+|\n+', str(text))
    # Keep only sentences with enough content
    sentences = [s.strip() for s in sentences if len(s.strip()) > 20]
    return sentences

def get_kbbert_embedding(text, tokenizer, model, max_length=512):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=True
    )
    with torch.no_grad():
        outputs = model(**inputs)
    token_embeddings = outputs.last_hidden_state.squeeze(0)
    attention_mask = inputs["attention_mask"].squeeze(0)
    mask_expanded = attention_mask.unsqueeze(-1).float()
    embedding = (token_embeddings * mask_expanded).sum(0) / mask_expanded.sum(0)
    return embedding.numpy()

# Build sentence-level corpus with place metadata
print("Generating sentence-level KB-BERT embeddings...")
sentence_records = []

for _, row in df.iterrows():
    sentences = split_into_sentences(row["text"])
    for sent in sentences:
        emb = get_kbbert_embedding(sent, tokenizer, model)
        sentence_records.append({
            "place":     row["place"],
            "sentence":  sent,
            "embedding": emb
        })

sent_df = pd.DataFrame(sentence_records)
sent_embeddings = np.array(sent_df["embedding"].tolist())

print(f"Total sentences extracted: {len(sent_df)}")
print(f"Embedding matrix: {sent_embeddings.shape}")
print(f"\nSentences per village:")
print(sent_df["place"].value_counts())

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
import umap

N_TOPICS = 5

# Normalise embeddings before clustering
sent_embeddings_norm = normalize(sent_embeddings)

# Reduce to 10D first (better than clustering in 768D)
print("Reducing dimensions with UMAP...")
reducer_topics = umap.UMAP(
    n_neighbors=min(10, len(sent_df) - 1),
    n_components=10,
    metric="cosine",
    random_state=42
)
sent_reduced = reducer_topics.fit_transform(sent_embeddings_norm)

# KMeans on reduced embeddings — more stable than HDBSCAN for topic modelling
print(f"Clustering into {N_TOPICS} topics...")
kmeans = KMeans(n_clusters=N_TOPICS, random_state=42, n_init=20)
sent_df["topic"] = kmeans.fit_predict(sent_reduced)

print("\nSentences per topic:")
print(sent_df["topic"].value_counts().sort_index())

Find the most representative sentences per topic by finding sentences closest to each cluster centroid

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

topic_centroids = kmeans.cluster_centers_

for topic_id in range(N_TOPICS):
    topic_mask = sent_df["topic"] == topic_id
    topic_sents = sent_df[topic_mask].copy()
    topic_embs = sent_reduced[topic_mask]

    # Rank by proximity to centroid
    centroid = topic_centroids[topic_id].reshape(1, -1)
    sims = cosine_similarity(topic_embs, centroid).flatten()
    topic_sents["centroid_sim"] = sims
    top_sents = topic_sents.nlargest(5, "centroid_sim")

    print(f"\nTopic {topic_id}  ({topic_mask.sum()} sentences)")
    print("-" * 55)
    for _, s in top_sents.iterrows():
        print(f"  [{s['place']}] {s['sentence'][:100]}")

    # Which villages dominate this topic?
    village_dist = topic_sents["place"].value_counts()
    print(f"\n  Village distribution: {dict(village_dist)}")

Village topic distribution — KB-BERT sentence clustering

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Count how many sentences each village has in each topic
topic_dist = sent_df.groupby(["place", "topic"]).size().unstack(fill_value=0)

# Normalise to proportions
topic_dist_norm = topic_dist.div(topic_dist.sum(axis=1), axis=0)

TOPIC_LABELS = {
    0: "Topic 0",   # e.g. "Entrepreneurship & economy"
    1: "Topic 1",   # e.g. "Nature & recreation"
    2: "Topic 2",   # e.g. "Community & services"
    3: "Topic 3",   # e.g. "Population & future"
    4: "Topic 4",   # e.g. "Land & agriculture"
}
topic_dist_norm.columns = [TOPIC_LABELS[c] for c in topic_dist_norm.columns]

print("Village topic distributions (KB-BERT derived):")
print(topic_dist_norm.round(3).to_string())

# Stacked bar chart
fig, ax = plt.subplots(figsize=(12, 6))
colors = ["#534AB7", "#0F6E56", "#BA7517", "#E24B4A", "#185FA5"]
bottom = np.zeros(len(topic_dist_norm))

for i, col in enumerate(topic_dist_norm.columns):
    ax.bar(
        topic_dist_norm.index,
        topic_dist_norm[col],
        bottom=bottom,
        color=colors[i],
        label=col,
        edgecolor="white",
        linewidth=0.5
    )
    bottom += topic_dist_norm[col].values

ax.set_title("Village topic distribution — KB-BERT sentence clustering",
             fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Topic proportion", fontsize=10)
ax.set_xlabel("Village", fontsize=10)
ax.set_ylim(0, 1)
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.tight_layout()
plt.savefig("kbbert_village_topics.png", dpi=150, bbox_inches="tight")
plt.show()

Define concern anchor sentences in Swedish, KB-BERT will find semantically similar sentences not just keyword matches. This is the key advantage over plain keyword counting

In [ ]:
CONCERN_ANCHORS = {
    "Infrastructure decay":  "Vägarna är dåliga och underhållet är eftersatt",
    "Service loss":          "Affären och skolan har stängt, servicen försvinner",
    "Population decline":    "Folk flyttar bort och byn blir allt mer avfolkad",
    "Youth emigration":      "Ungdomarna lämnar byn för att studera och jobba i staden",
    "Housing demand":        "Det behövs fler bostäder och tomter för inflyttning",
    "Digital exclusion":     "Internetuppkopplingen och bredbandet fungerar dåligt",
}

# Embed anchor sentences with KB-BERT
print("Embedding concern anchors...")
anchor_embeddings = {}
for concern, anchor_text in CONCERN_ANCHORS.items():
    anchor_embeddings[concern] = get_kbbert_embedding(
        anchor_text, tokenizer, model
    )

# Score each sentence against each concern anchor by cosine similarity
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

concern_scores = {c: [] for c in CONCERN_ANCHORS}

for emb in sent_embeddings:
    for concern, anchor_emb in anchor_embeddings.items():
        sim = cos_sim(emb.reshape(1, -1), anchor_emb.reshape(1, -1))[0][0]
        concern_scores[concern].append(float(sim))

# Add scores to sentence dataframe
for concern in CONCERN_ANCHORS:
    sent_df[f"concern_{concern}"] = concern_scores[concern]

# Aggregate per village — mean similarity to each concern
concern_cols = [f"concern_{c}" for c in CONCERN_ANCHORS]
village_concerns = sent_df.groupby("place")[concern_cols].mean()
village_concerns.columns = list(CONCERN_ANCHORS.keys())

print("\nConcern salience per village (KB-BERT semantic similarity):")
print(village_concerns.round(3).to_string())

Define Strength anchor sentences in Swedish, KB-BERT will find semantically similar sentences not just keyword matches.

In [ ]:
STRENGTH_ANCHORS = {
    "Nature & landscape":   "Naturen är vacker med skog, sjöar och friluftsliv",
    "Community cohesion":   "Vi hjälper varandra och gemenskapen i byn är stark",
    "Local economy":        "Det finns lokala företag och arbetstillfällen i byn",
    "Entrepreneurship":     "Det finns goda möjligheter att starta och driva företag",
    "Heritage & history":   "Byn har lång historia och starka kulturella traditioner",
    "Urban proximity":      "Det är nära till staden och lätt att pendla",
    "Services available":   "Det finns bra service med affär, skola och apotek",
}

print("Embedding strength anchors...")
strength_anchor_embeddings = {}
for strength, anchor_text in STRENGTH_ANCHORS.items():
    strength_anchor_embeddings[strength] = get_kbbert_embedding(
        anchor_text, tokenizer, model
    )

strength_scores = {s: [] for s in STRENGTH_ANCHORS}
for emb in sent_embeddings:
    for strength, anchor_emb in strength_anchor_embeddings.items():
        sim = cos_sim(emb.reshape(1, -1), anchor_emb.reshape(1, -1))[0][0]
        strength_scores[strength].append(float(sim))

for strength in STRENGTH_ANCHORS:
    sent_df[f"strength_{strength}"] = strength_scores[strength]

strength_cols_full = [f"strength_{s}" for s in STRENGTH_ANCHORS]
village_strengths = sent_df.groupby("place")[strength_cols_full].mean()
village_strengths.columns = list(STRENGTH_ANCHORS.keys())

print("\nStrength & asset framing per village (KB-BERT semantic similarity):")
print(village_strengths.round(3).to_string())

Visualize village characterisation via KB-BERT semantic anchoring

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(32, 10))

# Concerns heatmap
sns.heatmap(
    village_concerns.T,
    annot=True, fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.5,
    ax=axes[0],
    cbar_kws={"label": "Semantic similarity"}
)
axes[0].set_title("Concern salience\n(KB-BERT semantic similarity to concern anchors)",
                   fontsize=11, fontweight="bold")
axes[0].set_xlabel("Village")
axes[0].set_ylabel("Concern type")
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha="right")
plt.setp(axes[0].yaxis.get_majorticklabels(), rotation=0)

# Strengths heatmap
sns.heatmap(
    village_strengths.T,
    annot=True, fmt=".2f",
    cmap="YlGn",
    linewidths=0.5,
    ax=axes[1],
    cbar_kws={"label": "Semantic similarity"}
)
axes[1].set_title("Strength & asset framing\n(KB-BERT semantic similarity to strength anchors)",
                   fontsize=11, fontweight="bold")
axes[1].set_xlabel("Village")
axes[1].set_ylabel("Asset type")
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha="right")
plt.setp(axes[1].yaxis.get_majorticklabels(), rotation=0)

plt.suptitle("Village characterisation via KB-BERT semantic anchoring",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("kbbert_concerns_strengths.png", dpi=150, bbox_inches="tight")
plt.show()

# Additional visualizations

In [ ]:
topic_dist_norm.to_csv("kbbert_village_topics.csv")
village_concerns.to_csv("kbbert_village_concerns.csv")
village_strengths.to_csv("kbbert_village_strengths.csv")
sent_df.to_csv("kbbert_sentences_annotated.csv", index=False)

print("Saved:")
print("  kbbert_village_topics.csv         — topic distribution per village")
print("  kbbert_village_concerns.csv       — concern salience per village")
print("  kbbert_village_strengths.csv      — asset framing per village")
print("  kbbert_sentences_annotated.csv    — all sentences with scores")

  Bubble chart version of concern + strength profiles.
    Bubble SIZE = intensity (semantic similarity score)
    Bubble COLOR = concern (red scale) vs strength (green scale)
    One row per village easy to compare across villages at a glance.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

def plot_bubble_profile(concern_df, strength_df, save_path="kbbert_village_profile.png"):

    concerns  = list(concern_df.columns)
    strengths = list(strength_df.columns)
    villages  = concern_df.index.tolist()

    all_dims  = concerns + strengths
    n_dims    = len(all_dims)
    n_villages = len(villages)
    # Normalise scores 0–1 within each column so bubbles are comparable
    def norm(df):
        return (df - df.min()) / (df.max() - df.min() + 1e-9)

    concern_norm  = norm(concern_df)
    strength_norm = norm(strength_df)

    fig, ax = plt.subplots(figsize=(max(14, n_dims * 1.3), max(6, n_villages * 1.1)))
    ax.set_facecolor("#F8F8F6")
    fig.patch.set_facecolor("#F8F8F6")

    # Draw alternating column bands for readability
    for i in range(n_dims):
        color = "#EFF0EC" if i % 2 == 0 else "#F8F8F6"
        ax.axvspan(i - 0.5, i + 0.5, color=color, zorder=0)

    # Divider between concerns and strengths
    divider_x = len(concerns) - 0.5
    ax.axvline(divider_x, color="#CCCCCC", linewidth=1.5, linestyle="--", zorder=1)

    # Section labels
    ax.text(len(concerns) / 2 - 0.5, n_villages - 0.1,
            "⚠  CONCERNS", ha="center", va="bottom",
            fontsize=10, fontweight="bold", color="#C0392B",
            transform=ax.get_xaxis_transform())
    ax.text(len(concerns) + len(strengths) / 2 - 0.5, n_villages - 0.1,
            "✦  STRENGTHS", ha="center", va="bottom",
            fontsize=10, fontweight="bold", color="#1A7A4A",
            transform=ax.get_xaxis_transform())


        # Plot bubbles
    for v_idx, village in enumerate(villages):
        y = v_idx

        # Concern bubbles — red palette
        for c_idx, concern in enumerate(concerns):
            raw   = concern_df.loc[village, concern]
            norm_val = concern_norm.loc[village, concern]
            size  = norm_val * 1800 + 80   # min visible size
            alpha = 0.4 + norm_val * 0.55

            ax.scatter(c_idx, y, s=size,
                       color="#E24B4A", alpha=alpha,
                       edgecolors="white", linewidths=0.8, zorder=3)

            # Show raw score inside bubble if large enough
            if norm_val > 0.4:
                ax.text(c_idx, y, f"{raw:.2f}",
                        ha="center", va="center",
                        fontsize=7, color="white", fontweight="bold", zorder=4)

        # Strength bubbles — green palette
        for s_idx, strength in enumerate(strengths):
            raw   = strength_df.loc[village, strength]
            norm_val = strength_norm.loc[village, strength]
            size  = norm_val * 1800 + 80
            alpha = 0.4 + norm_val * 0.55
            x     = len(concerns) + s_idx

            ax.scatter(x, y, s=size,
                       color="#0F6E56", alpha=alpha,
                       edgecolors="white", linewidths=0.8, zorder=3)

            if norm_val > 0.4:
                ax.text(x, y, f"{raw:.2f}",
                        ha="center", va="center",
                        fontsize=7, color="white", fontweight="bold", zorder=4)

    # Axes formatting
    ax.set_xticks(range(n_dims))
    ax.set_xticklabels(all_dims, rotation=40, ha="right",
                        fontsize=9.5, color="#333333")
    ax.set_yticks(range(n_villages))
    ax.set_yticklabels(villages, fontsize=11, fontweight="500", color="#222222")
    ax.set_xlim(-0.6, n_dims - 0.4)
    ax.set_ylim(-0.7, n_villages - 0.3)
    ax.invert_yaxis()
    ax.tick_params(axis="both", length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

    # Grid lines (horizontal only, subtle)
    for v_idx in range(n_villages):
        ax.axhline(v_idx, color="#E0E0DA", linewidth=0.6, zorder=1)

    legend_elements = [
        mpatches.Patch(color="#E24B4A", alpha=0.75, label="Concern — larger = more prominent"),
        mpatches.Patch(color="#0F6E56", alpha=0.75, label="Strength — larger = more prominent"),
    ]
    ax.legend(handles=legend_elements, loc="lower right",
              fontsize=9, framealpha=0.9, edgecolor="#CCCCCC")

    ax.set_title(
        "Village characterisation via KB-BERT semantic anchoring\n"
        "Bubble size reflects how strongly each theme appears in village interviews",
        fontsize=13, fontweight="bold", pad=20, color="#111111"
    )

    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches="tight", facecolor="#F8F8F6")
    plt.show()
    print(f"Saved: {save_path}")


plot_bubble_profile(village_concerns, village_strengths,
                    save_path="kbbert_village_profile.png")

# Playing with the dataset

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import zscore
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from scipy.cluster.hierarchy import dendrogram, linkage

DARK   = "#ECEFF4"
PANEL  = "#FFFFFF"
GRID   = "#D8DEE9"
WHITE  = "#111827"
DIM    = "#4B5563"

COLORS = [
    "#4F46E5",
    "#059669",
    "#D97706",
    "#DC2626",
    "#2563EB",
    "#7C3AED",
    "#0F766E",
    "#BE123C"
]

def style_ax(ax, title, xlabel=None, ylabel=None):
    ax.set_facecolor(PANEL)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID)
    ax.tick_params(colors=DIM, labelsize=7.5)
    if xlabel: ax.set_xlabel(xlabel, color=DIM)
    if ylabel: ax.set_ylabel(ylabel, color=DIM)
    ax.set_title(title, color=WHITE, fontsize=9, fontweight="bold", pad=8)
    ax.grid(True, color=GRID, linewidth=0.5, alpha=0.6)

In [ ]:
# 1. UMAP Scatter
def plot_umap_clusters(umap_2d, cluster_labels, places, save_path=None):
    fig, ax = plt.subplots(figsize=(16,10))
    style_ax(ax, "UMAP Projection  ·  Village Embedding Space")

    for cid in sorted(set(cluster_labels)):
        mask = cluster_labels == cid
        color = "#555566" if cid == -1 else COLORS[cid % len(COLORS)]
        label = "Noise" if cid == -1 else f"Cluster {cid}"
        ax.scatter(umap_2d[mask,0], umap_2d[mask,1],
                   c=color, s=120, alpha=0.9,
                   edgecolors="white", linewidths=0.6,
                   label=label, zorder=3)

    for i, place in enumerate(places):
        ax.annotate(place, (umap_2d[i,0], umap_2d[i,1]),
                    fontsize=6.5, color=WHITE, alpha=0.85,
                    xytext=(4,4), textcoords="offset points")

    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.legend(fontsize=6.5, facecolor=PANEL, edgecolor=GRID, labelcolor=WHITE)

    if save_path:
        plt.savefig(save_path, dpi=180, bbox_inches="tight", facecolor=DARK)
    plt.show()



In [ ]:
 plot_umap_clusters(umap_2d, cluster_labels, places)

In [ ]:
#  2. Cosine Similarity
def plot_cosine_similarity(village_matrix, cluster_labels, places, save_path=None):
    fig, ax = plt.subplots(figsize=(16,10))
    style_ax(ax, "Cosine Similarity Matrix  ·  Reordered by Cluster")

    order = np.argsort(cluster_labels)
    ordered_places = [places[i] for i in order]

    sim_matrix = cosine_similarity(village_matrix)
    sim_ordered = sim_matrix[np.ix_(order, order)]

    im = ax.imshow(sim_ordered, cmap="magma", vmin=0.5, vmax=1.0, aspect="auto")

    ax.set_xticks(range(len(places)))
    ax.set_yticks(range(len(places)))
    ax.set_xticklabels(ordered_places, rotation=55, ha="right", fontsize=6.5, color=DIM)
    ax.set_yticklabels(ordered_places, fontsize=6.5, color=DIM)

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).ax.tick_params(colors=DIM, labelsize=6.5)
    ax.grid(False)

    # Draw cluster boundaries
    cluster_sizes = [np.sum(np.array(cluster_labels)[order] == c)
                     for c in sorted(set(cluster_labels))]
    boundary = 0
    for size in cluster_sizes[:-1]:
        boundary += size
        ax.axhline(boundary - 0.5, color="#7C6FE0", linewidth=1, alpha=0.6)
        ax.axvline(boundary - 0.5, color="#7C6FE0", linewidth=1, alpha=0.6)

    if save_path:
        plt.savefig(save_path, dpi=180, bbox_inches="tight", facecolor=DARK)
    plt.show()



In [ ]:
plot_cosine_similarity(village_matrix, cluster_labels, places)

In [ ]:
#  3. Embedding L2 Norm
def plot_embedding_norms(village_matrix, cluster_labels, places, save_path=None):
    fig, ax = plt.subplots(figsize=(16,10))
    style_ax(ax, "Embedding L2 Norm  ·  Per Village (corpus density proxy)")

    norms = np.linalg.norm(village_matrix, axis=1)
    bar_colors = [COLORS[l % len(COLORS)] if l != -1 else "#555566"
                  for l in cluster_labels]

    bars = ax.barh(places, norms, color=bar_colors,
                   edgecolor=PANEL, height=0.6)

    ax.axvline(norms.mean(), color="#F5A623", linewidth=1.2,
               linestyle="--", alpha=0.8, label=f"Mean = {norms.mean():.2f}")

    ax.set_xlabel("L2 norm")
    ax.legend(fontsize=7, facecolor=PANEL, edgecolor=GRID, labelcolor=WHITE)
    ax.tick_params(axis="y", labelsize=7, colors=WHITE)

    if save_path:
        plt.savefig(save_path, dpi=180, bbox_inches="tight", facecolor=DARK)
    plt.show()



In [ ]:
plot_embedding_norms(village_matrix, cluster_labels, places)

In [ ]:
#  4. Hierarchical Dendrogram
def plot_hierarchical_dendrogram(village_matrix, places, save_path=None):
    from scipy.cluster.hierarchy import dendrogram, linkage
    from scipy.spatial.distance import squareform

    fig, ax = plt.subplots(figsize=(16,10))
    style_ax(ax, "Hierarchical Clustering  ·  Ward Linkage on KB-BERT Embeddings")

    # Compute cosine similarity and distance
    sim_matrix = cosine_similarity(village_matrix)
    dist_matrix = 1 - sim_matrix
    np.fill_diagonal(dist_matrix, 0)

    # Condensed distance matrix for linkage
    condensed = squareform(dist_matrix)
    Z = linkage(condensed, method="ward")

    # Dendrogram
    dendrogram(
        Z, labels=places, ax=ax,
        leaf_rotation=55, leaf_font_size=7,
        color_threshold=0.3,
        above_threshold_color=DIM,
        link_color_func=lambda k: COLORS[k % len(COLORS)]
    )

    ax.set_ylabel("Distance", fontsize=8)
    for lbl in ax.get_xticklabels():
        lbl.set_color(WHITE)
    ax.tick_params(axis="y", colors=DIM)

    if save_path:
        plt.savefig(save_path, dpi=180, bbox_inches="tight", facecolor=DARK)
    plt.show()



In [ ]:
plot_hierarchical_dendrogram(village_matrix, places)

In [ ]:
#  5. Concern Z-Score Heatmap
def plot_concern_heatmap(concern_df, save_path=None):
    from scipy.stats import zscore

    fig, ax = plt.subplots(figsize=(16,10))
    style_ax(ax, "Concern Salience  ·  Z-score Normalised Across Villages")

    # Z-score normalization
    concern_z = concern_df.apply(zscore, axis=0).fillna(0)

    im = ax.imshow(concern_z.T.values, cmap="RdYlBu_r",
                   aspect="auto", vmin=-2, vmax=2)

    ax.set_xticks(range(len(concern_df)))
    ax.set_yticks(range(len(concern_df.columns)))
    ax.set_xticklabels(concern_df.index, rotation=45, ha="right",
                       fontsize=6.5, color=WHITE)
    ax.set_yticklabels(concern_df.columns, fontsize=6.5, color=WHITE)

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                 label="z-score").ax.tick_params(colors=DIM, labelsize=6)
    ax.grid(False)

    # Annotate each cell
    for i in range(len(concern_df)):
        for j in range(len(concern_df.columns)):
            val = concern_z.values[i, j]
            ax.text(i, j, f"{val:+.1f}", ha="center", va="center",
                    fontsize=5.5,
                    color="white" if abs(val) > 1 else DIM)

    if save_path:
        plt.savefig(save_path, dpi=180, bbox_inches="tight", facecolor=DARK)
    plt.show()



In [ ]:
plot_concern_heatmap(village_concerns)

In [ ]:
#  6. Strength Z-Score Heatmap
def plot_strength_heatmap(strength_df, save_path=None):
    from scipy.stats import zscore

    fig, ax = plt.subplots(figsize=(16,10))
    style_ax(ax, "Strength & Asset Framing  ·  Z-score Normalised")

    # Z-score normalization
    strength_z = strength_df.apply(zscore, axis=0).fillna(0)

    im = ax.imshow(strength_z.T.values, cmap="YlGn",
                   aspect="auto", vmin=-2, vmax=2)

    ax.set_xticks(range(len(strength_df)))
    ax.set_yticks(range(len(strength_df.columns)))
    ax.set_xticklabels(strength_df.index, rotation=45, ha="right",
                       fontsize=6.5, color=WHITE)
    ax.set_yticklabels(strength_df.columns, fontsize=6.5, color=WHITE)

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                 label="z-score").ax.tick_params(colors=DIM, labelsize=6)
    ax.grid(False)

    # Annotate each cell
    for i in range(len(strength_df)):
        for j in range(len(strength_df.columns)):
            val = strength_z.values[i, j]
            ax.text(i, j, f"{val:+.1f}", ha="center", va="center",
                    fontsize=5.5,
                    color="white" if abs(val) > 1 else DIM)

    if save_path:
        plt.savefig(save_path, dpi=180, bbox_inches="tight", facecolor=DARK)
    plt.show()



In [ ]:
plot_strength_heatmap(village_strengths)

In [ ]:
#  7. PCA Variance Explained
def plot_pca_variance(village_matrix, save_path=None):
    from sklearn.decomposition import PCA

    fig, ax = plt.subplots(figsize=(12,8))
    style_ax(ax, "PCA on KB-BERT Embeddings  ·  Variance Explained")

    pca = PCA(n_components=min(20, len(village_matrix)))
    pca.fit(village_matrix)

    indvar = pca.explained_variance_ratio_ * 100
    cumvar = np.cumsum(indvar)

    ax_twin = ax.twinx()

    ax.bar(range(1, len(indvar)+1), indvar,
           color="#7C6FE0", alpha=0.7, label="Per component")
    ax_twin.plot(range(1, len(cumvar)+1), cumvar,
                 color="#F5A623", linewidth=2, marker="o", markersize=4,
                 label="Cumulative")
    ax_twin.axhline(90, color="#E24B4A", linewidth=1, linestyle="--", alpha=0.7,
                    label="90% threshold")

    ax.set_xlabel("Principal component")
    ax.set_ylabel("Variance explained (%)")
    ax_twin.set_ylabel("Cumulative %", color="#F5A623", fontsize=8)
    ax_twin.tick_params(colors="#F5A623", labelsize=7)
    ax_twin.set_ylim(0, 105)

    # Legends
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax_twin.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2,
              fontsize=6.5, facecolor=PANEL, edgecolor=GRID, labelcolor=WHITE)
    ax_twin.spines["right"].set_edgecolor(GRID)

    if save_path:
        plt.savefig(save_path, dpi=180, bbox_inches="tight", facecolor=DARK)
    plt.show()



In [ ]:
plot_pca_variance(village_matrix)

In [ ]:
#  8. Sentence-level Topic Density
def plot_sentence_topic_density(sent_embeddings, sent_df, save_path=None):
    import umap as umap_lib
    from sklearn.preprocessing import normalize

    fig, ax = plt.subplots(figsize=(16,10))
    style_ax(ax, "Sentence Embedding Density  ·  UMAP 2D  ·  Coloured by Topic")

    if "topic" in sent_df.columns and len(sent_embeddings) > 10:
        reducer = umap_lib.UMAP(
            n_neighbors=min(8, len(sent_embeddings)-1),
            n_components=2, metric="cosine", random_state=42
        )
        sent_2d = reducer.fit_transform(normalize(sent_embeddings))
        unique_topics = sorted(sent_df["topic"].unique())

        for tid in unique_topics:
            mask = sent_df["topic"].values == tid
            ax.scatter(sent_2d[mask,0], sent_2d[mask,1],
                       c=COLORS[tid % len(COLORS)],
                       s=18, alpha=0.55, label=f"Topic {tid}",
                       edgecolors="none")

        ax.set_xlabel("UMAP-1")
        ax.set_ylabel("UMAP-2")
        ax.legend(fontsize=6.5, facecolor=PANEL,
                  edgecolor=GRID, labelcolor=WHITE,
                  markerscale=1.2, ncol=2)
    else:
        ax.text(0.5, 0.5, "Run preprocessing first\nto generate topic labels",
                ha="center", va="center", color=DIM,
                fontsize=9, transform=ax.transAxes)

    if save_path:
        plt.savefig(save_path, dpi=180, bbox_inches="tight", facecolor=DARK)
    plt.show()

In [ ]:
 plot_sentence_topic_density(sent_embeddings, sent_df)

In [ ]:
#  9. Concern vs Strength Scatter
def plot_concern_vs_strength(concern_df, strength_df, cluster_labels, places, save_path=None):
    fig, ax = plt.subplots(figsize=(16,10))
    style_ax(ax, "Concern vs Strength Balance  ·  Village Positioning")

    total_concern  = concern_df.sum(axis=1)
    total_strength = strength_df.sum(axis=1)

    for i, place in enumerate(concern_df.index):
        cid = cluster_labels[places.index(place)] if place in places else -1
        color = COLORS[cid % len(COLORS)] if cid != -1 else "#555566"

        ax.scatter(total_concern[place], total_strength[place],
                   s=160, color=color, alpha=0.9,
                   edgecolors="white", linewidths=0.8, zorder=3)


        ax.annotate(place, (total_concern[place], total_strength[place]),
                    fontsize=6.5, color=WHITE, alpha=0.9,
                    xytext=(5,5), textcoords="offset points")

    # Quadrant lines at medians
    ax.axvline(total_concern.median(), color=DIM,
               linewidth=0.8, linestyle="--", alpha=0.5)
    ax.axhline(total_strength.median(), color=DIM,
               linewidth=0.8, linestyle="--", alpha=0.5)

    # Quadrant labels
    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    ax.text(xlim[0] + 0.02*(xlim[1]-xlim[0]),
            ylim[1] - 0.05*(ylim[1]-ylim[0]),
            "Low concern\nHigh strength", fontsize=6.5,
            color="#2ECC8E", alpha=0.7, va="top")
    ax.text(xlim[1] - 0.02*(xlim[1]-xlim[0]),
            ylim[1] - 0.05*(ylim[1]-ylim[0]),
            "High concern\nHigh strength", fontsize=6.5,
            color="#F5A623", alpha=0.7, va="top", ha="right")
    ax.text(xlim[0] + 0.02*(xlim[1]-xlim[0]),
            ylim[0] + 0.05*(ylim[1]-ylim[0]),
            "Low concern\nLow strength", fontsize=6.5,
            color=DIM, alpha=0.7)
    ax.text(xlim[1] - 0.02*(xlim[1]-xlim[0]),
            ylim[0] + 0.05*(ylim[1]-ylim[0]),
            "High concern\nLow strength", fontsize=6.5,
            color="#E24B4A", alpha=0.7, ha="right")

    ax.set_xlabel("Total concern score (sum of semantic similarities)")
    ax.set_ylabel("Total strength score (sum of semantic similarities)")

    if save_path:
        plt.savefig(save_path, dpi=180, bbox_inches="tight", facecolor=DARK)
    plt.show()



In [ ]:
plot_concern_vs_strength(village_concerns, village_strengths, cluster_labels, places)

Other ways

In [ ]:
import plotly.graph_objects as go
import plotly.express as px

VILLAGE_COLORS = px.colors.qualitative.Vivid
TEXT_COLOR = "#111827"
DIM_COLOR  = "#4B5563"
PANEL_BG   = "#FFFFFF"
PAPER_BG   = "#ECEFF4"
GRID_COLOR = "#D8DEE9"

strength_df = village_strengths

villages = strength_df.index.tolist()
strength_cols = strength_df.columns.tolist()

fig = go.Figure()
for i, village in enumerate(villages):
    vals = strength_df.loc[village].tolist() + [strength_df.loc[village].tolist()[0]]
    cats = strength_cols + [strength_cols[0]]
    color = VILLAGE_COLORS[i % len(VILLAGE_COLORS)]
    fig.add_trace(go.Scatterpolar(
        r=vals, theta=cats,
        name=village,
        fill="toself",
        fillcolor=color,
        line=dict(color=color, width=2),
        opacity=0.65
    ))

fig.update_layout(
    title="Strength Profile per Village (Radar)",
    polar=dict(
        bgcolor=PANEL_BG,
        radialaxis=dict(color=DIM_COLOR, gridcolor=GRID_COLOR),
        angularaxis=dict(color=TEXT_COLOR, gridcolor=GRID_COLOR)
    ),
    paper_bgcolor=PAPER_BG,
    font=dict(color=TEXT_COLOR),
    height=1000, width=1600
)

fig.show()

In [ ]:
import plotly.graph_objects as go
import plotly.express as px

villages = strength_df.index.tolist()
dominant = strength_df.idxmax(axis=1)

asset_palette = {cat: px.colors.qualitative.Pastel[i % 10] for i, cat in enumerate(strength_df.columns)}
bar_colors = [asset_palette[dominant[v]] for v in villages]

fig = go.Figure(go.Bar(
    y=villages,
    x=[strength_df.loc[v, dominant[v]] for v in villages],
    orientation="h",
    marker_color=bar_colors,
    text=[dominant[v] for v in villages],
    textposition="inside",
    textfont=dict(size=9, color="white")
))

fig.update_layout(
    title="Dominant Asset by Village (Bar)",
    xaxis_title="KB-BERT similarity score",
    yaxis=dict(color=TEXT_COLOR),
    paper_bgcolor=PAPER_BG,
    plot_bgcolor=PANEL_BG,
    height=1000, width=1600
)

fig.show()

In [ ]:
import plotly.graph_objects as go

total_score = strength_df.sum(axis=1).sort_values(ascending=False)

fig = go.Figure(go.Bar(
    x=total_score.index,
    y=total_score.values,
    marker=dict(color=total_score.values, colorscale="Teal", showscale=True),
    text=[f"{v:.2f}" for v in total_score.values],
    textposition="outside"
))

fig.update_layout(
    title="Village Strength Ranking (Total Score)",
    xaxis_tickangle=-40,
    yaxis_title="Summed similarity",
    paper_bgcolor=PAPER_BG,
    plot_bgcolor=PANEL_BG,
    height=1000, width=1600
)

fig.show()

In [ ]:
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
import pandas as pd

strength_z = pd.DataFrame(StandardScaler().fit_transform(strength_df),
                          index=strength_df.index,
                          columns=strength_df.columns)

fig = go.Figure(go.Heatmap(
    z=strength_z.values,
    x=strength_z.columns,
    y=strength_z.index,
    colorscale="RdYlGn",
    zmid=0,
    zmin=-2.5,
    zmax=2.5
))

fig.update_layout(
    title="Z-score Heatmap",
    paper_bgcolor=PAPER_BG,
    plot_bgcolor=PANEL_BG,
    height=1000, width=1600
)

fig.show()

In [ ]:
import plotly.graph_objects as go
from sklearn.decomposition import PCA
import plotly.express as px

pca_coords = PCA(n_components=2).fit_transform(strength_df.values)
var_exp = PCA(n_components=2).fit(strength_df.values).explained_variance_ratio_ * 100
VILLAGE_COLORS = px.colors.qualitative.Vivid
dominant = strength_df.idxmax(axis=1)

fig = go.Figure()
for i, village in enumerate(strength_df.index):
    color = VILLAGE_COLORS[i % len(VILLAGE_COLORS)]
    fig.add_trace(go.Scatter(
        x=[pca_coords[i,0]],
        y=[pca_coords[i,1]],
        mode="markers+text",
        name=village,
        marker=dict(size=18, color=color, line=dict(width=1.5, color="white")),
        text=[village],
        textposition="top center"
    ))

fig.update_layout(
    title="PCA — Embedding Space",
    xaxis_title=f"PC1 ({var_exp[0]:.1f}% variance)",
    yaxis_title=f"PC2 ({var_exp[1]:.1f}% variance)",
    paper_bgcolor=PAPER_BG,
    plot_bgcolor=PANEL_BG,
    height=1000, width=1600
)

fig.show()